# ML Challenge Overfit et Debordés

## Import Packages

In [180]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor 
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder

## Data Importation

In [181]:
X_train = pd.read_csv("data/challenge_train_features.csv", index_col=0)
y_train = pd.read_csv("data/challenge_train_revenue.csv", index_col=0)
X_test = pd.read_csv("data/challenge_test_features.csv", index_col=0)

Transformations of data using pandas (for date) and for y_train (log transformation)

In [182]:
X_train["date_format"] = pd.to_datetime(X_train["date"], format="%m/%d/%y")
X_train.loc[X_train["date_format"].dt.year > 2025, "date_format"] -= pd.offsets.DateOffset(years=100)
X_train["year"] = X_train["date_format"].dt.year
X_train["month"] = X_train["date_format"].dt.month

X_test["date_format"] = pd.to_datetime(X_test["date"], format="%m/%d/%y")
X_test.loc[X_test["date_format"].dt.year > 2025, "date_format"] -= pd.offsets.DateOffset(years=100)
X_test["year"] = X_test["date_format"].dt.year
X_test["month"] = X_test["date_format"].dt.month

y_train_log = np.log1p(y_train.values)

Transformations of data using sklearn pipeline

In [ ]:

#Transformation functions


def clip_popularity(X):
    X = X.copy()
    X['popularity_score'] = X['popularity_score'].clip(upper=20)
    return X[['popularity_score']]

def budget_missing_indicator(X):
    X= X.copy()
    X['budget_is_zero'] = (X['budget'] == 0).astype(int)
    return X[['budget_is_zero']]

def log_budget(X):
    X = X.copy()
    X['budget'] = np.log1p(X['budget'].clip(lower=0))
    return X[['budget']]

def log_budget_with0(X):
    X= X.copy()
    X['budget_nonzero'] = X['budget'].replace(0, np.nan)
    X['log_budget'] = np.log1p(X['budget_nonzero'])
    return X[['log_budget']]

def collection_to_binary(X):
    return X.notna().astype(int).to_numpy().reshape(-1,1)

def english_to_binary(X):
    return (X == 'en').astype(int).to_numpy().reshape(-1,1)

def US_to_binary(X):
    return (X == 'US').astype(int).to_numpy().reshape(-1,1)


def budget_with_popularity(X):
    X = X.copy()
    X['budget_x_popularity'] = X['budget'] * X['popularity_score']
    return X[['budget_x_popularity']]

def get_first_genre(X):
    X = X.copy()
    X['genre'] = X['genre'].str.split(',').str[0]
    X['genre'] = X['genre'].fillna('Unknown')
    return X[['genre']]

def regrouper_genre(X):
    X = X.copy()
    haut = ["Animation","Adventure", "Family", "Fantasy", "Science Fiction", "Action"]
    
    
    X['grouped_genre'] = X['genre'].apply(lambda g: 'cat1' if g in haut else 'cat2')
    return X[['grouped_genre']]

# obtenir la liste des genres fréquents (>150 occurrences)
def get_frequent_genres(min_occurrence=150):
    X_train = X_train.copy()
    X_train["genres_list"] = X_train["genre"].apply(lambda x: x.split(",") if isinstance(x, str) else [])
    genre_counts = X_train["genres_list"].explode().value_counts()
    selected_genres = genre_counts[genre_counts >= min_occurrence].index.tolist()
    return selected_genres + ["Other"]


def encode_genres(X, min_occurrence = 150):
    X = X.copy()
    #Split genres into listts
    X["genres_list"] = X["genre"].apply(lambda x: x.split(",") if isinstance(x, str) else [])
    #compter les nombres de chaque genre et garder > 150
    genre_counts = X["genres_list"].explode().value_counts()
    selected_genres = genre_counts[genre_counts >= min_occurrence].index
    
    X["genres_list"] = X["genres_list"].apply(lambda lst: [g if g in selected_genres else "Other" for g in lst])
    
    for genre in selected_genres.tolist() + ["Other"]:
        X[genre] = X["genres_list"].apply(lambda lst: int(genre in lst))
        
    return X[selected_genres.tolist()+["Other"]]


# #pipeline for the variable "genre":
# genre_pipe = Pipeline([
#     ('get_first', FunctionTransformer(get_first_genre, validate=False)),
#     ('regrouper', FunctionTransformer(regrouper_genre, validate=False)),
#     ('onehot', OneHotEncoder(handle_unknown='ignore'))
# ])

### Pipeline


In [184]:
#We create the preprocessing pipeline
#Columns to be transformed
num_cols = ['budget', 'popularity_score']
cat_cols = ['collection', 'language', 'country','month']



preprocessor = ColumnTransformer(
    transformers=[
        # ('budget', 'passthrough', ['budget']),
        # ('budget_missing', FunctionTransformer(budget_missing_indicator, validate=False), ['budget']),
        ('log_budget', FunctionTransformer(log_budget_with0, validate=False), ['budget']),
        # ('budget_pop', FunctionTransformer(budget_with_popularity, validate=False), ['budget', 'popularity_score']),
        ('popularity','passthrough', ['popularity_score']),
        ('length','passthrough', ['length']),
        ('collection_bin', FunctionTransformer(collection_to_binary, validate=False), ['collection']),
        ('language_bin', FunctionTransformer(english_to_binary, validate=False), ['language']),
        ('country_bin', FunctionTransformer(US_to_binary, validate=False), ['country']),
        ('month_cat', OneHotEncoder(handle_unknown='ignore'), ['month']),
        ('genre', FunctionTransformer(encode_genres, validate=False), ['genre'])
        # ('genre_cat', genre_pipe, ['genre'])
    ],
    remainder='drop'
)


pipeline = Pipeline([
    ('preprocessor', preprocessor)
])

#We apply the pipeline to the training dataset
X_train_transformed = pipeline.fit_transform(X_train)
print(X_train_transformed.shape)
#We apply the same pipeline to the test dataset
X_test_transformed = pipeline.transform(X_test)
print(X_test_transformed.shape)


(2000, 30)
(500, 21)


In [ ]:
# Copie de ton dataset
X_train_proc = X_train.copy()
X_test_proc  = X_test.copy()

# 1️⃣ Traitement du budget
X_train_proc["log_budget"] = np.log1p(X_train_proc["budget"].clip(lower=0))
X_test_proc["log_budget"]  = np.log1p(X_test_proc["budget"].clip(lower=0))

# 2️⃣ Variable binaire pour la collection
X_train_proc["has_collection"] = X_train_proc["collection"].notna().astype(int)
X_test_proc["has_collection"]  = X_test_proc["collection"].notna().astype(int)

# 3️⃣ Variable binaire pour la langue
X_train_proc["is_en"] = (X_train_proc["language"] == "en").astype(int)
X_test_proc["is_en"]  = (X_test_proc["language"] == "en").astype(int)

# 4️⃣ Genres encodés manuellement (version simple et stable)
selected_genres = get_frequent_genres(X_train_proc, min_occurrence=150)
X_train_genres  = encode_genres_fixed(X_train_proc, selected_genres)
X_test_genres   = encode_genres_fixed(X_test_proc, selected_genres)

# 5️⃣ Fusion finale
X_train_final = pd.concat([X_train_proc[["popularity_score", "length", "log_budget", "has_collection", "is_en"]], X_train_genres], axis=1)
X_test_final  = pd.concat([X_test_proc[["popularity_score", "length", "log_budget", "has_collection", "is_en"]], X_test_genres], axis=1)

print(X_train_final.shape, X_test_final.shape)


Model

In [179]:
model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    min_child_weight=1.0,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_transformed, y_train_log)

y_pred_log= model.predict(X_test_transformed)

y_test_pred = np.expm1(y_pred_log).clip(0, None)


ValueError: Feature shape mismatch, expected: 30, got 21

Saving in a text file

In [ ]:
pred_str = ",".join([str(int(p)) for p in y_test_pred])  

with open("xgboost.txt", "w") as f:
    f.write(pred_str)

In [ ]:
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_log_error

# Catégories : ici aucune car toutes sont numériques (0/1)
train_pool = Pool(X_train_transformed, y_train_log)

model = CatBoostRegressor(
    iterations=800,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',  # on apprend sur log(y)
    random_seed=2,
    verbose=100
)

model.fit(train_pool)

# Prédictions
y_pred_log = model.predict(X_test_transformed)
y_pred = np.expm1(y_pred_log) 

0:	learn: 2.8520681	total: 9.7ms	remaining: 7.75s
100:	learn: 1.8836201	total: 162ms	remaining: 1.12s
200:	learn: 1.7414098	total: 316ms	remaining: 943ms
300:	learn: 1.6376232	total: 473ms	remaining: 785ms
400:	learn: 1.5364375	total: 620ms	remaining: 616ms
500:	learn: 1.4616386	total: 771ms	remaining: 460ms
600:	learn: 1.4003085	total: 921ms	remaining: 305ms
700:	learn: 1.3412472	total: 1.08s	remaining: 152ms
799:	learn: 1.2870169	total: 1.26s	remaining: 0us


In [ ]:
pred_str = ",".join([str(int(p)) for p in y_pred])

with open("catboost.txt", "w") as f:
    f.write(pred_str)

y_pred = 0